In [ ]:
import requests
import pandas as pd
from io import StringIO
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta.tables import DeltaTable

doaj_api_key = dbutils.secrets.get(scope="doaj", key="api_key")
response = requests.get(f'https://doaj.org/csv?api_key={doaj_api_key}', allow_redirects=True)
response.raise_for_status()

csv_content = response.content.decode('utf-8')
pandas_df = pd.read_csv(StringIO(csv_content))

# standardize column headers
pandas_df.columns = [col.lower().replace(' ', '_').replace('(', '').replace(')', '').replace('?', '').replace('-', '_') for col in pandas_df.columns]

# convert to dataframe
df = spark.createDataFrame(pandas_df)
df = df.withColumnRenamed('when_did_the_journal_start_to_publish_all_content_using_an_open_license', 'oa_start_year')
df = df.withColumn("oa_start_year", F.col("oa_start_year").cast("int"))

# combine issns into array and remove nulls
df = df.withColumn(
"issns",
    F.expr("filter(array(journal_issn_print_version, journal_eissn_online_version), x -> x is not null and trim(x) != '')")
)

# update in a safe way, merging data from the CSV rather than overwriting everything, every time
target_table = "openalex.sources.doaj_from_csv"

if spark.catalog.tableExists(target_table):        
    print("delta table exists, updating")
    target = DeltaTable.forName(spark, target_table)

    target_columns = set(spark.table(target_table).columns)
    common_columns = [col for col in df.columns if col in target_columns]

    result_df = (
        target.alias("target")
        .merge(
            df.alias("source"),
            "target.url_in_doaj = source.url_in_doaj"
        )
        .whenMatchedUpdate(
            condition="source.last_updated_date > target.last_updated_date",
            set={col: F.col(f"source.{col}") for col in common_columns}
        )
        .whenNotMatchedInsert(
            values={col: F.col(f"source.{col}") for col in common_columns}
        )
        .execute()
    )

    print("Merge result:")
    result = result_df.collect()[0]
    print(f"Rows inserted: {result['num_inserted_rows']}")
    print(f"Rows updated: {result['num_updated_rows']}")
else:
    # first-time write
    df.write.format("delta").saveAsTable(target_table)
    print("Initial table write complete.")